In [1]:
pip install pyogrio


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/jupyter/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
GIS Property Intelligence — Address-Driven Spatial RAG
Supported input formats: FileGDB, Shapefile, CSV, GeoJSON, JSON
Vector Store: Milvus (Self-Hosted)
"""

import json
import math
import os
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from pymilvus import MilvusClient
from pydantic import BaseModel, Field, field_validator
from typing import Any
from sklearn.neighbors import BallTree
import gc

# ── 1. Configuration ──────────────────────────────────────────────────────────
OLLAMA_HOSTS = [
    "http://10.10.10.100:11434",
    "https://ollama.splsystems.in",
]

def resolve_ollama_host(hosts: list[str]) -> str:
    for host in hosts:
        try:
            resp = requests.get(host, timeout=5)
            if resp.status_code < 500:
                return host
        except requests.exceptions.RequestException:
            continue
    return hosts[-1]

OLLAMA_HOST = resolve_ollama_host(OLLAMA_HOSTS)
EMBED_MODEL = "nomic-embed-text-v2-moe:latest"
LLM_MODEL   = "gemma4:latest"
MAX_RECORDS = 5000
TOP_K       = 10   

# Milvus Configuration
MILVUS_URI   = "http://10.10.10.130:19530"
COLLECTION_NAME = "address_points"
VECTOR_DIM   = 768  
MILVUS_TOP_K = 5    

FILE_PATH    = "Address_Points.geojson" 

COLUMN_ALIASES: dict[str, list[str]] = {
    "address":   ["address", "addr", "full_address", "site_address", "ADDRNAME"],
    "city":      ["city", "municipality", "COMMUNITY"],
    "state":     ["state", "st"], 
    "zipcode":   ["zipcode", "zip", "postal_code", "ADDRZIP"],
    "latitude":  ["latitude", "lat", "y", "POINT_Y"],
    "longitude": ["longitude", "lon", "lng", "x", "POINT_X"],
}

EARTH_RADIUS_MILES = 3_958.8

# Global state variables
validated = []
tree = None
milvus_client = None


# ── 2. Pre-Flight Diagnostics ─────────────────────────────────────────────────
def run_pre_flight_checks():
    print("[DEBUG] Running Pre-Flight Infrastructure Checks...")
    
    def check_endpoint(url: str, name: str):
        try:
            resp = requests.get(url, timeout=5)
            print(f"[DEBUG] [OK] {name} is reachable (Status: {resp.status_code})")
        except requests.exceptions.RequestException as e:
            print(f"[DEBUG] [FAIL] {name} is unreachable at {url}. Error: {e}")

    check_endpoint(OLLAMA_HOST, "Ollama Server")
    check_endpoint("https://nominatim.openstreetmap.org/status.php?format=json", "Nominatim Geocoder")
    print("-" * 50)


# ── 3. Pydantic Models ────────────────────────────────────────────────────────
class AddressPoint(BaseModel):
    address:    str
    city:       str
    state:      str
    zipcode:    str
    latitude:   float
    longitude:  float
    attributes: dict[str, Any] = Field(default_factory=dict)

    @field_validator("address", "city", "state", "zipcode", mode="before")
    @classmethod
    def coerce_str(cls, v: Any) -> str:
        return str(v) if pd.notna(v) and v is not None else ""

    @field_validator("latitude", "longitude", mode="before")
    @classmethod
    def coerce_float(cls, v: Any) -> float:
        try:
            f = float(v)
            return f if math.isfinite(f) else 0.0
        except (TypeError, ValueError):
            return 0.0


# ── 4. Spatial Indexing & Geocoding Helpers ───────────────────────────────────
def spatial_query(lat: float, lon: float, k: int = TOP_K) -> list[dict]:
    if not tree or not validated:
        return []
    query = np.array([[math.radians(lat), math.radians(lon)]])
    distances, indices = tree.query(query, k=min(k, len(validated)))

    results = []
    for dist_rad, idx in zip(distances[0], indices[0]):
        rec = validated[idx]
        rec["distance_miles"] = round(dist_rad * EARTH_RADIUS_MILES, 4)
        results.append(rec)
    return results

def geocode(address: str) -> tuple[float, float] | None:
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "GIS-Property-Intelligence/1.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        hits = resp.json()
        if hits:
            return float(hits[0]["lat"]), float(hits[0]["lon"])
        print(f"[DEBUG] Geocode returned no results for: {address}")
    except Exception as exc:
        print(f"[DEBUG] [ERROR] Geocode failed: {exc}")
    return None


# ── 5. Embeddings & Milvus Helpers ────────────────────────────────────────────
def build_geo_document(point: dict, rank: int) -> str:
    attrs = point.get("attributes", {})
    return (
        f"Address: {point.get('address','')}, {point.get('city','')} {point.get('zipcode','')}\n"
        f"Distance: {point.get('distance_miles', 'N/A')} miles\n"
        f"Zoning: {attrs.get('zoning', 'Unknown')}\n"
    ).strip()

def embed(texts: list[str]) -> list[list[float]]:
    url = f"{OLLAMA_HOST}/api/embed"
    body = {"model": EMBED_MODEL, "input": texts}
    try:
        resp = requests.post(url, json=body, timeout=120)
        resp.raise_for_status()
        return resp.json()["embeddings"]
    except Exception as e:
        print(f"[DEBUG] [ERROR] Embedding API failure: {e}")
        return [[0.0]*VECTOR_DIM for _ in texts]


# ── 6. Optimized Data Loading Pipeline ────────────────────────────────────────
def load_and_normalize_efficiently(filepath: str, chunk_size: int = 1000):
    print(f"[DEBUG] Processing dataset: {filepath}")
    
    try:
        if filepath.endswith(('.geojson', '.shp', '.gdb')):
            df = gpd.read_file(filepath, engine="pyogrio")
            if df.geometry is not None and 'latitude' not in df.columns:
                df["latitude"] = df.geometry.y
                df["longitude"] = df.geometry.x
            df = df.drop(columns=['geometry']) 
        else:
            df = pd.read_csv(filepath)

        if MAX_RECORDS:
            df = df.head(MAX_RECORDS)

        rename_map = {}
        for canonical, aliases in COLUMN_ALIASES.items():
            for alias in aliases:
                rename_map[alias] = canonical
                rename_map[alias.lower()] = canonical
                rename_map[alias.upper()] = canonical
        df.rename(columns=rename_map, inplace=True)

        for col in ["address", "city", "state", "zipcode", "latitude", "longitude"]:
            if col not in df.columns:
                df[col] = "" if col not in ["latitude", "longitude"] else 0.0

        mapped_cols = list(COLUMN_ALIASES.keys())
        unmapped_cols = [c for c in df.columns if c not in mapped_cols]
        df['attributes'] = df[unmapped_cols].apply(lambda row: row.to_dict(), axis=1)
        df = df.drop(columns=unmapped_cols)

        for start in range(0, len(df), chunk_size):
            chunk = df.iloc[start:start + chunk_size]
            yield chunk.to_dict(orient="records")
            del chunk
            gc.collect()

    except Exception as e:
        print(f"[DEBUG] [FATAL] Data loading failed: {e}")
        yield []

def process_and_insert_pipeline(filepath: str):
    global milvus_client, tree, validated
    milvus_client = MilvusClient(uri=MILVUS_URI)
    
    if milvus_client.has_collection(collection_name=COLLECTION_NAME):
        milvus_client.drop_collection(collection_name=COLLECTION_NAME)

    milvus_client.create_collection(
        collection_name=COLLECTION_NAME,
        dimension=VECTOR_DIM,
        metric_type="COSINE" 
    )

    global_id_counter = 0
    all_valid_coords = []      
    all_valid_metadata = []    

    for chunk in load_and_normalize_efficiently(filepath, chunk_size=500):
        if not chunk: continue
        
        valid_batch = []
        for rec in chunk:
            try:
                point = AddressPoint(**rec)
                if point.latitude != 0.0 or point.longitude != 0.0:
                    valid_batch.append(point)
            except Exception:
                continue 
        
        if not valid_batch: continue

        docs = [build_geo_document(p.model_dump(), i) for i, p in enumerate(valid_batch)]
        vectors = embed(docs)
        
        insert_data = []
        for i, point in enumerate(valid_batch):
            insert_data.append({
                "id": global_id_counter,
                "vector": vectors[i],
                "document": docs[i],
                "address": point.address,
                "latitude": point.latitude,
                "longitude": point.longitude
            })
            
            all_valid_coords.append([math.radians(point.latitude), math.radians(point.longitude)])
            all_valid_metadata.append(point.model_dump())
            global_id_counter += 1
            
        if insert_data:
            milvus_client.insert(collection_name=COLLECTION_NAME, data=insert_data)
            print(f"[DEBUG] Inserted batch. Total records so far: {global_id_counter}")

    validated = all_valid_metadata 
    if all_valid_coords:
        tree = BallTree(np.array(all_valid_coords, dtype=np.float64), metric="haversine")
        print(f"[DEBUG] BallTree spatial index built for {len(all_valid_coords)} coordinates.")


# ── 7. RAG Controller ─────────────────────────────────────────────────────────
def ask_address(address: str, question: str, verbose: bool = False) -> str:
    print(f"\n[DEBUG] --- Starting RAG Pipeline for query: '{question}' ---")
    
    coords = geocode(address)
    if coords is None:
        return f"[ERROR] Could not geocode: '{address}'"
    lat, lon = coords
    if verbose: print(f"[DEBUG] Geocoded {address} -> Lat: {lat}, Lon: {lon}")

    nearby = spatial_query(lat, lon, k=TOP_K)
    if verbose: print(f"[DEBUG] Found {len(nearby)} nearby features via BallTree.")

    context = {"coordinates": {"lat": lat, "lon": lon}, "nearest": nearby[0] if nearby else None}
    
    q_vec = embed([question])[0]
    
    search_res = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[q_vec],
        limit=MILVUS_TOP_K,
        output_fields=["document"],
        search_params={"metric_type": "COSINE"}
    )
    
    retrieved = []
    if search_res and len(search_res) > 0:
        for hit in search_res[0]:
            retrieved.append(hit["entity"]["document"])
            
    if verbose: print(f"[DEBUG] Retrieved {len(retrieved)} semantic documents from Milvus.")

    prompt = f"""You are a GIS Property Intelligence Assistant.
Address: {address}
Coordinates: ({lat:.6f}, {lon:.6f})
Question: {question}

=== Spatial Context ===
{json.dumps(context, indent=2)}

=== Retrieved Documents ===
{"\n---\n".join(retrieved)}

Instructions:
- Answer the question using ONLY the supplied context and documents.
- Be specific — cite distances and zone codes where available.
- If information is unavailable, state that clearly.
"""

    if verbose: print("[DEBUG] Sending prompt to Gemma4...")
    url = f"{OLLAMA_HOST}/api/generate"
    body = {"model": LLM_MODEL, "prompt": prompt, "stream": False}
    
    try:
        resp = requests.post(url, json=body, timeout=120)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        return f"[DEBUG] [ERROR] LLM Generation failed: {e}"


# ── 8. Main Execution ─────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=== Initializing Memory-Optimized GIS RAG System ===")
    
    # 1. Check endpoints
    run_pre_flight_checks()
    
    # 2. Run the chunked streaming pipeline (handles load, validate, index, insert)
    process_and_insert_pipeline(FILE_PATH)
    
    print("\n=== System Ready. Running Test Queries ===")
    
    # Example 1
    ans1 = ask_address(
        "949 Sapphire St",
        "What are the closest address points to this location?",
        verbose=True
    )
    print(f"\nResponse 1:\n{ans1}\n")

    # Example 2
    ans2 = ask_address(
        "949 Sapphire St",
        "Which community, ZIP code, and parcel information are nearby?",
        verbose=True
    )
    print(f"\nResponse 2:\n{ans2}\n")

=== Initializing Memory-Optimized GIS RAG System ===
[DEBUG] Running Pre-Flight Infrastructure Checks...
[DEBUG] [OK] Ollama Server is reachable (Status: 200)
[DEBUG] [OK] Nominatim Geocoder is reachable (Status: 200)
--------------------------------------------------
[DEBUG] Processing dataset: Address_Points.geojson
[DEBUG] Inserted batch. Total records so far: 500
[DEBUG] Inserted batch. Total records so far: 1000
[DEBUG] Inserted batch. Total records so far: 1500
[DEBUG] Inserted batch. Total records so far: 2000
[DEBUG] Inserted batch. Total records so far: 2500
[DEBUG] Inserted batch. Total records so far: 3000
[DEBUG] Inserted batch. Total records so far: 3500
[DEBUG] Inserted batch. Total records so far: 4000
[DEBUG] Inserted batch. Total records so far: 4500
[DEBUG] Inserted batch. Total records so far: 5000
[DEBUG] BallTree spatial index built for 5000 coordinates.

=== System Ready. Running Test Queries ===

[DEBUG] --- Starting RAG Pipeline for query: 'What are the closest 